In [1]:
# Parameters
flux_method = "summed_map"
dig_mode = "no_dig"


In [2]:
from pathlib import Path
import sys

_here = Path.cwd().resolve()
_candidates = (_here, *_here.parents)
REPO_ROOT = next((candidate for candidate in _candidates if (candidate / "m33_pipeline").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate repo root from {Path.cwd()}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from m33_pipeline.notebook_setup import prepare_notebook

REPO_ROOT = prepare_notebook(REPO_ROOT)
print(f"Notebook working directory set to: {REPO_ROOT}")

Notebook working directory set to: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1


In [3]:
flux_method = 'summed_map'
dig_mode = 'no_dig'

from m33_pipeline.config import get_derived_config
from m33_pipeline.derived import (
    add_clustering_metrics,
    add_electron_density,
    add_logU_KK04,
    add_metallicity_columns,
    add_metallicity_error_columns,
    add_symmetry_class,
    merge_field_flux_catalogs,
    write_clustering_outputs,
    write_combined_catalog,
    write_derived_stage_catalog,
    write_total_flux_catalog,
)
from m33_pipeline.validate import validate_total_catalog


# Merge per-field flux catalogs


In [4]:
derived_config = get_derived_config()
all_catalog = merge_field_flux_catalogs(method=flux_method, dig_mode=dig_mode)
output_path = write_total_flux_catalog(all_catalog, method=flux_method, dig_mode=dig_mode)
print("Flux method:", flux_method)
print("DIG mode:", dig_mode)
print("Combined catalog shape:", all_catalog.shape)
print("Saved combined catalog to:", output_path)
# validate_total_catalog(all_catalog)


Flux method: summed_map
DIG mode: no_dig
Combined catalog shape: (1189, 88)
Saved combined catalog to: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/summed_map/no_dig/total_flux_catalog.csv


# Add ionization parameter


In [5]:
cat = add_logU_KK04(all_catalog.copy(), n_mc=derived_config.logu_n_mc, seed=123, metallicity_cal="M13_O3N2")
derived_output_path = write_derived_stage_catalog(cat, "ionization_parameter", method=flux_method, dig_mode=dig_mode)
print("Number of columns:", len(cat.columns))
print("Saved combined catalog with ionization parameter:", derived_output_path)


Number of columns: 103
Saved combined catalog with ionization parameter: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/summed_map/no_dig/derived/total_flux_catalog_ionization_parameter.csv


# Add electron density


In [6]:
df = add_electron_density(cat.copy(), n_mc=derived_config.density_n_mc)
derived_output_path = write_derived_stage_catalog(df, "density", method=flux_method, dig_mode=dig_mode)
print("Number of columns:", len(df.columns))
print("Saved combined catalog with electron densities:", derived_output_path)


Number of columns: 109
Saved combined catalog with electron densities: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/summed_map/no_dig/derived/total_flux_catalog_density.csv


# Add symmetry classification


In [7]:
df = add_symmetry_class(df.copy())
derived_output_path = write_derived_stage_catalog(df, "other_derived", method=flux_method, dig_mode=dig_mode)
print("Symmetry classification counts:")
print(df["symmetry_class"].value_counts())
print("Number of columns:", len(df.columns))
print("Saved combined catalog with other derived properties:", derived_output_path)


Symmetry classification counts:
symmetry_class
asymmetric    1007
symmetric      182
Name: count, dtype: int64
Number of columns: 110
Saved combined catalog with other derived properties: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/summed_map/no_dig/derived/total_flux_catalog_other_derived.csv


# Add metallicity calibrations


In [8]:
df = add_metallicity_columns(df.copy())
df = add_metallicity_error_columns(df.copy(), n_mc=derived_config.metallicity_n_mc, seed=123)
derived_output_path = write_derived_stage_catalog(df, "metallicity", method=flux_method, dig_mode=dig_mode)
combined_output_path = write_combined_catalog(df, method=flux_method, dig_mode=dig_mode)
print("Number of columns:", len(df.columns))
print("Saved combined catalog with metallicities:", derived_output_path)
print("Saved final combined catalog:", combined_output_path)


Number of columns: 162
Saved combined catalog with metallicities: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/summed_map/no_dig/derived/total_flux_catalog_metallicity.csv
Saved final combined catalog: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/summed_map/no_dig/total_flux_catalog_combined.csv


# Add deprojected clustering metrics


In [9]:
clustered_df, global_stats, ripley_df, pcf_df = add_clustering_metrics(df.copy())
outputs = write_clustering_outputs(clustered_df, global_stats, ripley_df, pcf_df, method=flux_method, dig_mode=dig_mode)
print("Saved catalog:", outputs["catalog"])
print("Saved global stats:", outputs["global"])
print("Saved Ripley profile:", outputs["ripley"])
print("Saved pair-correlation profile:", outputs["pcf"])


Saved catalog: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/summed_map/no_dig/derived/total_flux_catalog_clustering.csv
Saved global stats: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/summed_map/no_dig/derived/clustering_global_statistics.csv
Saved Ripley profile: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/summed_map/no_dig/derived/clustering_ripley_profile.csv
Saved pair-correlation profile: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/summed_map/no_dig/derived/clustering_pair_correlation_profile.csv
